# Plate detection — A1 (YOLO26n vs YOLOv8n)
Thin driver (OS-independent). All logic lives in `plate_detect`; this notebook only calls the CLI.

- **Local kernel** with a GPU: repo + prepared `data/` already on disk → the Setup cell just `pip install`s the package.
- **Colab runtime** (incl. VSCode-driven, headless): the Setup cell clones the private repo (GitHub PAT via prompt), installs the CLI, and pulls raw A1 from a Drive-shared `A1.zip` via `gdown` (set `DRIVE_FILE_ID`). All later cells run from the repo root.

> **Headless caveat:** `drive.mount()` and Colab Secrets need the Colab **web** frontend — they fail when the runtime is driven from VSCode (400 Bad Request / cancelled auth). So the PAT is entered at a `getpass` prompt and data comes via `gdown`, not a Drive mount.

In [1]:
# === Setup ===
# LOCAL kernel: repo already on disk + data prepared -> just install the package:
#     %cd /path/to/UIT2026-DoAnCuoiKi
#     !pip install -e src/ml/plate_detection_pipeline
# COLAB runtime (incl. VSCode-driven, HEADLESS): clones private repo BRANCH, installs CLI,
#   pulls raw A1 from a Drive-shared zip via gdown.
#   NOTE: drive.mount() and google.colab Secrets DO NOT WORK from VSCode — they need the Colab
#   web frontend. So: PAT via getpass prompt, data via gdown (no mount, no secrets).
import os, glob, shutil, getpass

IN_COLAB = "google.colab" in str(get_ipython())
REPO     = "/content/UIT2026-DoAnCuoiKi"
BRANCH   = "feat/plate-detect-a1"              # package NOT merged to main yet — clone this branch
RAW      = "data/raw/kaggle_vn_plate_segment"  # layout the A1Adapter expects: {images,labels}/{train,val}
# MyDrive/UIT_2025/datasets/A1.zip, shared "Anyone with the link" (id from the share URL):
DRIVE_FILE_ID = "1hwIns2lhAgg3i9gKdSccVuMil-AmZZty"

if IN_COLAB:
    # 1) clone the PRIVATE repo, feature branch. PAT via prompt (input box appears in VSCode/Colab).
    if not os.path.isdir(REPO):
        tok = getpass.getpass("GitHub PAT: ")
        !git clone --branch {BRANCH} --single-branch https://{tok}@github.com/UIT-DoAnCuoiKi/UIT2026-DoAnCuoiKi.git {REPO}
        del tok
    assert os.path.isdir(REPO), "clone failed — PAT lacks read access to the org repo (see README 'PAT setup')"
    %cd {REPO}
    !git rev-parse --abbrev-ref HEAD   # confirm the feature branch is checked out

    # 2) install the package -> puts the `plate_detect` CLI on PATH
    !pip install -q -e src/ml/plate_detection_pipeline

    # 3) raw A1 from the Drive zip via gdown (headless; NO drive.mount). Symlink RAW to it.
    if not os.path.isdir(f"{RAW}/images/train"):
        assert DRIVE_FILE_ID, "set DRIVE_FILE_ID above (share A1.zip 'Anyone with link', copy id from URL)"
        !pip install -q gdown
        !gdown "https://drive.google.com/uc?id={DRIVE_FILE_ID}" -O /tmp/A1.zip
        !unzip -q -o /tmp/A1.zip -d /tmp/a1
        hits = glob.glob("/tmp/a1/**/images/train", recursive=True)
        assert hits, "images/train not found after unzip — inspect /tmp/a1 and adjust"
        root = os.path.abspath(hits[0][: -len("/images/train")])
        os.makedirs(os.path.dirname(RAW), exist_ok=True)
        if os.path.islink(RAW) or os.path.exists(RAW):
            (os.unlink if os.path.islink(RAW) else shutil.rmtree)(RAW)
        os.symlink(root, os.path.abspath(RAW))
else:
    # local kernel: assume cwd is the repo root and data/ already present
    !pip install -q -e src/ml/plate_detection_pipeline

# 4) sanity-check the raw layout the adapter reads (train + val, images + labels)
for s in ("train", "val"):
    for k in ("images", "labels"):
        assert os.path.isdir(f"{RAW}/{k}/{s}"), f"missing {RAW}/{k}/{s} — check zip split names (val vs valid)"
print("OK — CLI installed, raw A1 ready at", RAW)

Cloning into '/content/UIT2026-DoAnCuoiKi'...
remote: Enumerating objects: 905, done.
remote: Counting objects: 100% (327/327), done.
remote: Compressing objects: 100% (171/171), done.
remote: Total 905 (delta 183), reused 250 (delta 145), pack-reused 578 (from 1)
Receiving objects: 100% (905/905), 37.78 MiB | 33.12 MiB/s, done.
Resolving deltas: 100% (475/475), done.
/content/UIT2026-DoAnCuoiKi
feat/plate-detect-a1
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 127.9 MB/s eta 0:00:0000:010:01
  Building editable for plate_detect (pyproject.toml) ... done
Downloading...
From (original): https://drive.google.com/uc?id=1hwIns2lhAgg3i9gKdSccVuMil-AmZZty
From (redirected): https://drive.google.com/uc

In [2]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

CUDA: True NVIDIA A100-SXM4-80GB


## 1. Prepare (class-map gate → split → dedup train↔test & train↔val → validate)

In [3]:
!plate_detect prepare

prepared: {'counts': {'train': 3433, 'val': 573, 'test': 572}, 'dup_train_test': 230, 'dup_train_val': 226, 'class_map': {1: 'bien_2hang', 0: 'bien_1hang'}, 'phash_report': 'data/processed/a1_det/phash_report.txt'}


In [4]:
!plate_detect check

data-contract OK


## 2. Train — full matrix @640 (both models × seeds 0,1,2)

In [5]:
# QUICK SMOKE TEST (10 epochs, 1 seed via configs/quick.yaml). For real run drop --config and use: --imgsz 640 --seeds 0,1,2
!plate_detect train --config src/ml/plate_detection_pipeline/configs/quick.yaml --project runs

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.116 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=src/ml/plate_detect/configs/a1_det.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=T

## 3. imgsz ablation @960 (single seed, both models)

In [6]:
# QUICK SMOKE TEST (10 epochs, 1 seed). --imgsz 960 overrides quick.yaml. For real run drop --config and use: --imgsz 960 --seeds 0
!plate_detect train --config src/ml/plate_detection_pipeline/configs/quick.yaml --imgsz 960 --project runs

New https://pypi.org/project/ultralytics/8.4.116 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=src/ml/plate_detect/configs/a1_det.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, 

## 4. Export best → ONNX (per model & imgsz), parity-checked

In [7]:
# example; repeat per model/imgsz best run:
!plate_detect export --weights runs/yolo26n_s0_640/weights/best.pt --out weights/yolo26n_a1_640.onnx --imgsz 640

Traceback (most recent call last):
  File "/usr/local/bin/plate_detect", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/content/UIT2026-DoAnCuoiKi/src/ml/plate_detection_pipeline/plate_detect/cli.py", line 168, in main
    out = cmd_export(cfg, a.weights, a.out, a.imgsz or cfg.imgsz)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/UIT2026-DoAnCuoiKi/src/ml/plate_detection_pipeline/plate_detect/cli.py", line 107, in cmd_export
    export(weights_pt, out_onnx, imgsz=imgsz)
  File "/content/UIT2026-DoAnCuoiKi/src/ml/plate_detection_pipeline/plate_detect/export/to_onnx.py", line 28, in export
    model = YOLO(weights_pt)
            ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/yolo/model.py", line 76, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 144, in __init__
    self._load(model, 

## 5. Evaluate on A1 test → comparison table + experiments.csv

In [8]:
!plate_detect eval --imgszs 640,960 --project runs --weights-dir weights --sample-image data/processed/a1_det/images/test/$(ls data/processed/a1_det/images/test | head -1)

| model | imgsz | mAP@0.5 (mean±std) | mAP@0.5:0.95 [95% CI] | P | R | params(M) | FLOPs(G) | size(MB) | lat_model(ms) | lat_e2e(ms) | FPS |
|---|---|---|---|---|---|---|---|---|---|---|---|


## 6. Package results → Drive (or browser download)

Zips `runs/`, `weights/`, and `experiments.csv` into one archive. Prefers a **Drive** mount (survives the runtime timeout, no size limit); falls back to a **browser download** when Drive can't mount (VSCode-driven headless — see the Setup caveat) or when running a local kernel.

In [9]:
# === Package all results into one .zip, then Drive-copy or browser-download ===
import os, glob, datetime, shutil

STAMP   = datetime.datetime.now().strftime("%Y%m%d_%H%M")
ARCHIVE = f"/content/plate_det_results_{STAMP}.zip" if IN_COLAB else f"plate_det_results_{STAMP}.zip"

# collect what exists (runs/ = weights+plots+curves, weights/ = exported ONNX, experiments.csv)
targets = [p for p in ("runs", "weights", "experiments.csv") if os.path.exists(p)]
assert targets, "nothing to zip — run train/export/eval first"
print("zipping:", targets)
!zip -rq "{ARCHIVE}" {" ".join(targets)}
print("archive:", ARCHIVE, f"({os.path.getsize(ARCHIVE)/1e6:.1f} MB)")

# 1) preferred: copy into Drive (persists past the runtime timeout). mount fails on VSCode-headless.
saved = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        dest = "/content/drive/MyDrive/UIT_2025/results"
        os.makedirs(dest, exist_ok=True)
        shutil.copy(ARCHIVE, dest)
        print("copied to Drive:", os.path.join(dest, os.path.basename(ARCHIVE)))
        saved = True
    except Exception as e:
        print("Drive mount unavailable (VSCode-headless?), falling back to download:", repr(e))

# 2) fallback: browser download (Colab web) — for local kernels the file is already on disk.
if not saved:
    if IN_COLAB:
        from google.colab import files
        files.download(ARCHIVE)
    else:
        print("saved locally at:", os.path.abspath(ARCHIVE))

zipping: ['runs', 'weights']
archive: /content/plate_det_results_20260809_1632.zip (59.2 MB)
Drive mount unavailable (VSCode-headless?), falling back to download: ValueError('mount failed')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>